### Which searching algrithms are giving best results in our case?

In [1]:
!pip install qdrant-client -q
!pip install fastembed-gpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.6 MB/s eta 0:00:00


In [2]:
import json
import pandas as pd

from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding

In [3]:
encoder = TextEmbedding()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

model_optimized.onnx:   0%|          | 0.00/66.5M [00:00<?, ?B/s]

In [3]:
client = QdrantClient(location=":memory:")

In [ ]:
TextEmbedding.list_supported_models()

In [14]:
for models in TextEmbedding.list_supported_models():
    if models['dim'] == 512:
        print(models)

{'model': 'BAAI/bge-small-zh-v1.5', 'sources': {'hf': 'Qdrant/bge-small-zh-v1.5', 'url': 'https://storage.googleapis.com/qdrant-fastembed/fast-bge-small-zh-v1.5.tar.gz', '_deprecated_tar_struct': True}, 'model_file': 'model_optimized.onnx', 'description': 'Text embeddings, Unimodal (text), Chinese, 512 input tokens truncation, Prefixes for queries/documents: not so necessary, 2023 year.', 'license': 'mit', 'size_in_GB': 0.09, 'additional_files': [], 'dim': 512, 'tasks': {}}
{'model': 'Qdrant/clip-ViT-B-32-text', 'sources': {'hf': 'Qdrant/clip-ViT-B-32-text', 'url': None, '_deprecated_tar_struct': False}, 'model_file': 'model.onnx', 'description': 'Text embeddings, Multimodal (text&image), English, 77 input tokens truncation, Prefixes for queries/documents: not necessary, 2021 year', 'license': 'mit', 'size_in_GB': 0.25, 'additional_files': [], 'dim': 512, 'tasks': {}}
{'model': 'jinaai/jina-embeddings-v2-small-en', 'sources': {'hf': 'xenova/jina-embeddings-v2-small-en', 'url': None, '_

In [24]:
model_handle = "jinaai/jina-embeddings-v2-small-en"

In [25]:
EMBEDDING_DIMENSIONALITY = 512

1. collection
2. points -> model name
3. push the points in collection
4. search query

In [14]:
def build_collection(name_collection: str, EMBEDDING_DIMENSION: int):

    try:
        if name_collection in client.get_collections().collections:
            print(f"Collection '{name_collection}' already exists.")
            return

        client.create_collection(
            collection_name=name_collection,
            vectors_config=models.VectorParams(
                size=EMBEDDING_DIMENSION,
                distance=models.Distance.COSINE
            )
        )

        print(f"Qdrant collection '{name_collection}' created (dimension: {EMBEDDING_DIMENSION})")

    except Exception as e:
        print(f"Failed to create collection '{name_collection}': {e}")


In [16]:
build_collection('Testing', 512)

❌ Failed to create collection 'Testing': Collection Testing already exists


In [15]:
build_collection('Testing2', 512)

✅ Qdrant collection 'Testing2' created (dimension: 512)


In [23]:
from typing import List

def populate_collection(name_collection: str, name_model: str, documents: List[dict]):
    points = []

    for record in documents:

        text_to_embed = f"{record['term']}: {record['definition']} {record['extra']}"

        point = models.PointStruct(
            id = record['id'],
            vector= models.Document(text = text_to_embed, model = name_model),
            payload={
                'term': record['term'],
                'about': f"{record['definition']} {record['extra']}"
            }
        )

        points.append(point)

    client.upsert(
        collection_name=name_collection,
        points=points
    )

    print(f"Collection '{name_collection}' filled with {len(points)} records.")

In [18]:
import json

with open('Data/data.json', 'rt') as f_in:
    documents = json.load(f_in)

In [25]:
populate_collection('Testing2', 'jinaai/jina-embeddings-v2-small-en', documents=documents)

✅ Collection 'Testing2' filled with 829 records.


In [54]:
def search(question):

    results = client.query_points(
        collection_name = collection,
        query = models.Document(text = question, model=model_handle),
        limit = 5,
        with_payload=True
    )

    best_matches = []
    for scored_point in results.points:
        best_matches.append(scored_point.id)

    return best_matches

In [55]:
search('What the hell is going on here?')

['66a82179a8e816dd929a66653fe743bc',
 '0b55a3eb0b1e56bc962e9c615c9d1815',
 '9ed9cfd8b11f1e8dd180dea172d2820d',
 '43ccf9204f999859f46111a2831464df',
 '434d045111a36ab0fbabbc4d4c686da6']

In [56]:
import pandas as pd

In [57]:
ground_truth_df = pd.read_csv('ground_truth.csv')
ground_truth_df.head()

,question,id
0,"What does the phrase ""10-1"" typically signify ...",f7808776ba3a73de7de01d677d9b2185
1,Imagine you're a camera assistant and you urge...,f7808776ba3a73de7de01d677d9b2185
2,Why is it important for film crews to have spe...,f7808776ba3a73de7de01d677d9b2185
3,"If a director hears ""10-1"" over the walkie-tal...",f7808776ba3a73de7de01d677d9b2185
4,"Besides a bathroom break, what other private a...",f7808776ba3a73de7de01d677d9b2185


In [58]:
ground_truth_df['semantic_search'] = ground_truth_df['question'].apply(search)

In [82]:
def score_hit_rate(output, true):

    hits = 0
    for id in output:
        if id == true:
            hits += 1
            break

    return hits

def score_mrr(ouput, true):

    weight = 1.0
    score = 0.0

    for index, id in enumerate(ouput):
        if id == true:
            score += (weight)

        weight /= (index + 1)

    return score

In [84]:
total_hits = 0

for i in range(len(ground_truth_df)):
    total_hits += score_hit_rate(ground_truth_df.iloc[i]['semantic_search'], ground_truth_df.iloc[i]['id'])

In [85]:
total_hits / len(ground_truth_df)

0.7225572979493365

In [86]:
total_hits_mrr = 0

for i in range(len(ground_truth_df)):
    total_hits_mrr += score_mrr(ground_truth_df.iloc[i]['semantic_search'], ground_truth_df.iloc[i]['id'])

In [88]:
total_hits_mrr / len(ground_truth_df)

0.6751005227181331